# 1. Setup

3D keeps the two frozen DINOv2 ViT-B/14 towers and two primitive classifiers of frozen 3A. It applies a DeCov-inspired penalty to crop and disease logits, cross-crop supervised contrastive learning to disease embeddings, and mild Gaussian blur to training images. The frozen setup matches 3A's epoch budget and avoids introducing a backbone fine-tuning change.

In [ ]:
import gc, json, sys
from pathlib import Path
from datetime import datetime
import timm, torch
import pandas as pd
from torch import nn
from torchvision import transforms
from torch.nn import functional as F
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'common/dataset.py').is_file())
sys.path.insert(0, str(ROOT))
from common import config as cfg
from common.dataset import get_loaders, get_pairs, get_final_test_loaders
from common.evaluate import evaluate, val_accuracy, concept_accuracy
MODEL = 'vit_base_patch14_dinov2.lvd142m'
MODEL_NAME, FAMILY = 'approach3d_dino_logit_decov_supcon_blur', 'D'
DEBUG = True
HP = dict(backbone=MODEL, batch_size=8 if DEBUG else 16, epochs=1 if DEBUG else 20,
          crop_lr=1e-4, disease_lr=3e-4, weight_decay=1e-4, label_smoothing=0.1,
          seed=cfg.SEED, norm='imagenet', hidden_dim=256, feature_dim=128,
          supcon_temperature=0.1, dino_revision='main',
          blur_probability=0.30, blur_kernel=9, blur_sigma=[0.1, 2.0])
EPOCHS = HP['epochs']
cfg.NUM_WORKERS = 0  # Exact ToTnki runtime setting; common files are unchanged.
cfg.set_seed(cfg.SEED)
RUN = MODEL_NAME + ('_debug' if DEBUG else '') + '_' + datetime.now().strftime('%Y%m%d_%H%M%S_%f')
OUT = cfg.CHECKPOINT_DIR / RUN
environment = dict(python=sys.version, torch=str(torch.__version__), timm=str(timm.__version__), device=str(cfg.DEVICE))

if not DEBUG:
    assert (HP['epochs'],HP['batch_size'],HP['crop_lr'],HP['disease_lr'],HP['weight_decay'],HP['label_smoothing']) == (20,16,1e-4,3e-4,1e-4,0.1), 'Keep the ToTnki baseline training budget.'

# 2. Data

Use the common split and image transforms. Insert a 9×9 Gaussian blur before tensor conversion on the training dataset only, using 30% probability and sigma sampled between 0.1 and 2.0. The kernel size and torchvision implementation are local choices. [Huan et al., Section 2.3](https://doi.org/10.3390/electronics14173410)

In [ ]:
pairs = get_pairs()
assert pairs.shape == (cfg.NUM_SEEN_PAIRS, 2)


def loaders():
    cfg.set_seed(cfg.SEED)
    train, val, seen, unseen = get_loaders(HP['batch_size'], norm=HP['norm'], debug=DEBUG)
    steps = list(train.dataset.transform.transforms)
    tensor_step = next(i for i, step in enumerate(steps) if isinstance(step, transforms.ToTensor))
    blur = transforms.GaussianBlur(HP['blur_kernel'], sigma=tuple(HP['blur_sigma']))
    steps.insert(tensor_step, transforms.RandomApply([blur], p=HP['blur_probability']))
    train.dataset.transform = transforms.Compose(steps)
    train.pin_memory = val.pin_memory = False
    return train, val, seen, unseen


train_loader, val_loader, _, _ = loaders()
assert not set(train_loader.dataset.frame.image) & set(val_loader.dataset.frame.image)
sample_counts = dict(train=len(train_loader.dataset), val=len(val_loader.dataset))
print('Device:', cfg.DEVICE, 'Data:', cfg.DATA_DIR, 'Samples:', sample_counts)

# 3. Model

The classification objective combines crop and disease cross-entropy with two regularizers.

- **Logit decorrelation** penalizes the mean squared cross-correlation of crop logits and disease logits across the batch. This adapts DeCov from hidden-unit covariance to standardized logits across two heads. [Cogswell et al.](https://arxiv.org/pdf/1511.06068)
- **Cross-crop SupCon** pulls together disease embeddings with the same disease across different crops. Other diseases act as negatives. Same-disease, same-crop samples are excluded from the denominator, and anchors need both a positive and a negative. Uses the dataset's known crop labels to select pairs within a batch. [Zhang et al.](https://proceedings.mlr.press/v162/zhang22z.html) · [Khosla et al.](https://arxiv.org/abs/2004.11362)

The related [unseen-crop study](https://doi.org/10.3390/electronics15081553) motivates disease-invariant features under a different evaluation setting. Here the test shift is new combinations of crop and disease concepts.

In [ ]:
def snapshot(repository,revision,allow_download,patterns):
    from huggingface_hub import snapshot_download
    path=Path(snapshot_download(repository,revision=revision,allow_patterns=patterns,local_files_only=not allow_download))
    return path

class Dino(nn.Module):
    """Exact two-independent-tower initialization from ToTnki's released runner."""
    def __init__(self,config,device,allow_download=False,revision=None):
        super().__init__()
        import timm
        path=snapshot('timm/vit_base_patch14_dinov2.lvd142m',revision or config['dino_revision'],allow_download,
                      ['config.json','model.safetensors','pytorch_model.bin'])
        weights=next(path/name for name in ['model.safetensors','pytorch_model.bin'] if (path/name).is_file())
        def pinned_source_model(classes):
            # Same source factory, with only an explicit weight-file binding added.
            model=timm.create_model(MODEL,pretrained=True,num_classes=classes,img_size=cfg.IMAGE_SIZE,
                                    pretrained_cfg_overlay={'file':str(weights)})
            return model
        self.crop=pinned_source_model(cfg.NUM_CROPS)
        self.disease=pinned_source_model(cfg.NUM_DISEASES)
        self.augmentation_rng_state=torch.get_rng_state().clone()
        self.width=self.crop.num_features
        self.metadata=dict(repository='timm/vit_base_patch14_dinov2.lvd142m',revision=path.name,
            architecture=MODEL,width=self.width,independent_visual_towers=2,
            initialization='ToTnki source factory settings, crop then disease, explicitly pinned public weight file',
            preprocessing='common imagenet; img_size224; timm native position resampling and pre_logits')
        self.to(device)
    @torch.no_grad()
    def features(self,pixels):
        return (self.crop.forward_head(self.crop.forward_features(pixels),pre_logits=True).float(),
                self.disease.forward_head(self.disease.forward_features(pixels),pre_logits=True).float())
    def forward(self,pixels):return self.features(pixels)

def projection(width,hidden,out):return nn.Sequential(nn.Linear(width,hidden),nn.ReLU(),nn.Linear(hidden,out))


RECIPE = dict(sc=0.1, dis=0.01)


def cross_crop_supcon(features, crops, diseases):
    same_disease = diseases[:, None] == diseases[None, :]
    positives = same_disease & (crops[:, None] != crops[None, :])
    negatives = ~same_disease
    valid = positives.any(1) & negatives.any(1)
    if not valid.any():
        return features.sum() * 0, 0.
    features = F.normalize(features.float(), dim=1)
    logits = features[valid] @ features.T / HP['supcon_temperature']
    keep = (positives | negatives)[valid]
    positive = positives[valid]
    normalizer = torch.logsumexp(logits.masked_fill(~keep, -torch.inf), dim=1, keepdim=True)
    log_prob = (logits - normalizer).masked_fill(~positive, 0)
    loss = -(log_prob.sum(1) / positive.sum(1)).mean()
    return loss, valid.float().mean().item()


def logit_decov(crop_logits, disease_logits):
    if crop_logits.shape[0] < 2:
        return (crop_logits.sum() + disease_logits.sum()) * 0
    def standardize(logits):
        centered = logits.float() - logits.float().mean(0)
        return centered / (centered.square().mean(0) + 1e-6).sqrt()
    correlation = standardize(crop_logits).T @ standardize(disease_logits) / len(crop_logits)
    return correlation.square().mean()


class Model(nn.Module):
    def __init__(self, backbone):
        super().__init__()
        self.backbone = backbone
        width, hidden = backbone.width, HP['hidden_dim']
        dim = HP['feature_dim']
        self.crop_projection = projection(width, hidden, dim)
        self.disease_projection = projection(width, hidden, dim)

        self.crop_head = nn.Linear(dim, cfg.NUM_CROPS)
        self.disease_head = nn.Linear(dim, cfg.NUM_DISEASES)

    def forward(self, pixels):
        crop_features, disease_features = self.backbone(pixels)
        crop = F.normalize(self.crop_projection(crop_features), dim=1)
        disease = F.normalize(self.disease_projection(disease_features), dim=1)
        return self.crop_head(crop), self.disease_head(disease), crop, disease



def loss(output, labels):
    crop_logits, disease_logits, _, disease_features = output
    crops, diseases = pairs.to(labels.device)[labels].unbind(1)
    smooth = HP['label_smoothing']
    classification = (F.cross_entropy(crop_logits, crops, label_smoothing=smooth)
                      + F.cross_entropy(disease_logits, diseases, label_smoothing=smooth))
    contrastive, anchors = cross_crop_supcon(disease_features, crops, diseases)
    decov = logit_decov(crop_logits, disease_logits)
    return classification + RECIPE['sc'] * contrastive + RECIPE['dis'] * decov, anchors


def make_model(metadata=None):
    cfg.set_seed(cfg.SEED)
    metadata = metadata or {}
    backbone = Dino(HP, cfg.DEVICE, allow_download=True, revision=metadata.get('dino', {}).get('revision'))
    backbone.requires_grad_(False).eval()
    cfg.set_seed(cfg.SEED)
    model = Model(backbone).to(cfg.DEVICE)
    torch.set_rng_state(backbone.augmentation_rng_state)
    return model, dict(dino=backbone.metadata)



def make_predict(model, candidate_pairs):
    crop_ids, disease_ids = candidate_pairs.to(cfg.DEVICE).unbind(1)

    @torch.no_grad()
    def predict(images):
        model.eval()
        crop, disease = model(images.to(cfg.DEVICE))[:2]
        return crop.log_softmax(1)[:, crop_ids] + disease.log_softmax(1)[:, disease_ids]
    return predict

# 4. Training
Train one complete approach using the baseline training budget and common validation for checkpoint selection. No official test data is accessed.


In [ ]:
OUT.mkdir(parents=True, exist_ok=False)
cfg.set_seed(cfg.SEED)
train_loader, val_loader, _, _ = loaders()
for loader in (train_loader, val_loader): loader.pin_memory = False
model, metadata = make_model()
recipe = RECIPE
crop_params = []; disease_params = []
for name, parameter in model.named_parameters():
    if parameter.requires_grad:
        (disease_params if 'disease' in name else crop_params).append(parameter)
optimizers = [torch.optim.AdamW(parameters, lr=lr, weight_decay=HP['weight_decay'])
              for parameters, lr in [(crop_params, HP['crop_lr']), (disease_params, HP['disease_lr'])] if parameters]
schedules = [torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS) for opt in optimizers]
history = []; best_val = -1.; checkpoint_path = OUT / (MODEL_NAME + '.pt')
for epoch in range(EPOCHS):
    model.train(); model.backbone.eval()
    total = 0.; samples = 0; anchor_fraction = 0.
    for images, _, _, labels in train_loader:
        for opt in optimizers: opt.zero_grad(set_to_none=True)
        values = model(images.to(cfg.DEVICE))
        value, anchors = loss(values, labels.to(cfg.DEVICE))
        if not torch.isfinite(value): raise RuntimeError('Nonfinite loss.')
        value.backward()
        if any(p.grad is not None for p in model.backbone.parameters() if not p.requires_grad):
            raise RuntimeError('Frozen DINO received gradients.')
        gradients = [p.grad for p in model.parameters() if p.requires_grad and p.grad is not None]
        if not gradients or not all(torch.isfinite(g).all() for g in gradients):
            raise RuntimeError('Invalid training gradients.')
        for opt in optimizers: opt.step()
        total += value.item() * len(labels); samples += len(labels)
        anchor_fraction += anchors * len(labels)
    for schedule in schedules: schedule.step()
    predict = make_predict(model, pairs)
    validation = val_accuracy(predict, val_loader)
    history.append(dict(epoch=epoch+1, loss=total/samples, val_top1=validation,
                        valid_crosscrop_anchor_fraction=anchor_fraction/samples))
    print(history[-1])
    if validation > best_val:
        best_val = validation
        torch.save(dict(model_name=MODEL_NAME,
            state_dict={k:v.detach().cpu() for k,v in model.state_dict().items()},
            hp=HP, debug=DEBUG, family=FAMILY, recipe=recipe,
            pairs=pairs, metadata=metadata, best_val=best_val,
            best_epoch=epoch+1, environment=environment), checkpoint_path)
del predict, model, optimizers, schedules
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()
selection = dict(model_name=MODEL_NAME, family=FAMILY, hp=HP, debug=DEBUG,
    recipe=RECIPE, checkpoint=checkpoint_path.name, best_val=best_val, history=history,
    train_images=train_loader.dataset.frame.image.tolist(),
    validation_images=val_loader.dataset.frame.image.tolist(),
    development_pairs=pairs.tolist(), criterion='common validation Top1; earliest epoch on ties')
with (OUT/'selection.json').open('x', encoding='utf8') as file: json.dump(selection, file, indent=2)
print('Selection frozen:', OUT/'selection.json')

# 5. Evaluation
Same common.evaluate as the old CLIP notebook. After ALL choices are frozen, enable RUN_FINAL_EVALUATION. Official333 candidates appear only here. The two extra component accuracies use the SAME clean unseen predictions, not robustness outputs; they describe unseen-combination test images, not genuinely unseen crop/disease classes.


In [ ]:
RUN_FINAL_EVALUATION = False  # Enable only after all model choices are frozen.
reports = []
if RUN_FINAL_EVALUATION:
    if DEBUG:
        raise ValueError('Official evaluation requires a full non-DEBUG run.')
    run_folder = OUT
    selection = json.loads((run_folder / 'selection.json').read_text(encoding='utf8'))
    checkpoint = torch.load(run_folder / selection['checkpoint'], map_location='cpu', weights_only=True)
    expected = dict(model_name=MODEL_NAME, family=FAMILY, debug=False, hp=HP, recipe=RECIPE)
    if any(record.get(key) != value for record in (selection, checkpoint)
           for key, value in expected.items()):
        raise ValueError('The selected run or checkpoint does not match the frozen settings.')
    if selection['development_pairs'] != pairs.tolist() or not torch.equal(checkpoint['pairs'], pairs):
        raise ValueError('Development candidates changed.')
    if (selection['train_images'] != train_loader.dataset.frame.image.tolist()
            or selection['validation_images'] != val_loader.dataset.frame.image.tolist()):
        raise ValueError('Common train/validation membership changed.')
    model, metadata = make_model(checkpoint['metadata'])
    if metadata != checkpoint['metadata']:
        raise ValueError('Public pretrained model identity changed.')
    model.load_state_dict(checkpoint['state_dict'])
    model.eval().requires_grad_(False)
    # Only now open the official candidates and test loaders.
    final_pairs = get_pairs(final_evaluation=True)
    seen_loader, unseen_loader = get_final_test_loaders(
        HP['batch_size'], norm=HP['norm'], final_evaluation=True)
    sample_counts.update(seen=len(seen_loader.dataset), unseen=len(unseen_loader.dataset))
    predict_final = make_predict(model, final_pairs)
    # common.evaluate predicts clean seen, then clean unseen, then robustness.
    seen_count = sample_counts['seen']
    clean_end = seen_count + sample_counts['unseen']
    unseen_batches, processed = [], 0
    def predict(images):
        global processed
        scores = predict_final(images)
        if seen_count <= processed < clean_end:
            unseen_batches.append(scores.float().cpu())
        processed += len(images)
        return scores
    RESULTS = cfg.RESULTS_DIR / run_folder.name
    RESULTS.mkdir(parents=True, exist_ok=False)
    previous_results = cfg.RESULTS_DIR
    try:
        cfg.RESULTS_DIR = RESULTS
        result = evaluate(predict, seen_loader, unseen_loader, MODEL_NAME, HP, robust=True)
    finally:
        cfg.RESULTS_DIR = previous_results
    unseen_labels = torch.as_tensor(unseen_loader.dataset.frame.pair_row.to_numpy())
    crop_acc, disease_acc = concept_accuracy(torch.cat(unseen_batches), unseen_labels, final_evaluation=True)
    result.update(unseen_test_crop_acc=crop_acc, unseen_test_disease_acc=disease_acc,
                  debug=DEBUG, best_val=checkpoint['best_val'])
    reports.append(result)
    del predict_final, model, unseen_batches

# 6. Save
Same summary fields as the old CLIP notebook. Separate run folders prevent overwriting baselines. Keep checkpoints and result JSON/CSV/confusion files. One common seed does not establish statistical significance; formal results are not generated during authoring.


In [ ]:
if RUN_FINAL_EVALUATION:
    for result in reports:
        (RESULTS/(result['model']+'.json')).write_text(json.dumps(result, indent=2), encoding='utf8')
    summary = dict(selection=selection, results=reports, hp=HP,
                   environment=environment, samples=sample_counts)
    (RESULTS/(MODEL_NAME+'_comparison.json')).write_text(json.dumps(summary, indent=2), encoding='utf8')
    table = pd.DataFrame(reports)
    table.to_csv(RESULTS/(MODEL_NAME+'_comparison.csv'), index=False)
    display(table[['model','seen_top1','unseen_top1','harmonic_mean',
                   'unseen_test_crop_acc','unseen_test_disease_acc']])
    print('Checkpoints:', run_folder, 'Results:', RESULTS)
else:
    print('Save the complete checkpoint run folder after training:', OUT)
